# **FULL PIPELINE**
### *Truncation Impact Score: A Diagnostic Metric Framework for Quantifying Salient Content Loss in Long-Form Biomedical Summarization*

This document specifies a reproducible experimental workflow for abstractive summarization on the PubMed corpus. The notebook implements deterministic dataset sampling, corpus-level statistics, Truncation Impact Score (TIS) computation, model configuration and parameter-efficient fine-tuning (LoRA) for BART and LED variants, batched inference, and stratified evaluation by truncation difficulty. Generated artifacts—including trained adapters, per-bin evaluation reports, and diagnostic analyses (token distributions, TIS summaries, and directory-level artifact listings)—support downstream quantitative analyses and figure generation.

In [ ]:
# Imports & project setup
import sys
from pathlib import Path
from datetime import datetime
import json
import logging
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from IPython.display import display, HTML

# Project root and path setup.
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Local project modules
from scripts.data import DatasetRecord, dataset_statistics, load_pubmed_split, sample_records
from scripts.text import oracle_summary, simple_sentence_split, simple_tokenize, token_count, sentence_tfidf_score
from scripts.tis import CorpusStatistics, TISResult, build_corpus_statistics, compute_tis, summarize_tis
from scripts.models import BART_SPEC, LED_SPEC, HuggingFaceSummarizer
from scripts.metrics import aggregate_metric_sets, summary_metrics
from scripts.analysis import BinThresholds, evaluate_by_tis_bins
from scripts.training import run_training_pipeline

# Standard library prints for reproducibility & quick checks.
print("Consolidated imports loaded.")
print(f"Project root: {PROJECT_ROOT}")
print("Project modules imported.")

## **Phase 1: Setup & Global Configuration**

*Establishes the experimental environment, filesystem layout, and authoritative pipeline constants (sampling sizes, token cutoffs, LoRA defaults, optimizer settings, and profiling options). All hardcoded values are declared here as a single source of truth.*

In [ ]:
# Filesystem layout, directories, and global experiment constants
# Centralized authoritative configuration for the notebook (paths, model cutoffs,
# LoRA and batch parameters, profiling constants, and TIS bins). Keeps outputs
# reproducible and easy to change from a single location.

# Filesystem layout and directories
FINE_TUNED_MODELS_DIR = PROJECT_ROOT / "fine_tuned_models"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
DATASET_ROOT = PROJECT_ROOT / "PubMed_DATASET"

# Ensure artifact directories exist
for directory in [
    FINE_TUNED_MODELS_DIR,
    ARTIFACTS_DIR,
    REPORTS_DIR,
    FINE_TUNED_MODELS_DIR / "bart_large",
    FINE_TUNED_MODELS_DIR / "led_large",
    FINE_TUNED_MODELS_DIR / "profile_bart",
    FINE_TUNED_MODELS_DIR / "profile_led",
]:
    directory.mkdir(parents=True, exist_ok=True)

# Logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("pubmed_tis_pipeline")

# Global experiment constants (single authoritative source).
PIPELINE_VERSION = "1.0"
TRAIN_SAMPLES = 12000
VAL_SAMPLES = 500
TEST_SAMPLES = 1500

CUTOFF_TOKENS_BART = 1024
CUTOFF_TOKENS_LED = 16384
TARGET_SUMMARY_LENGTH = 256
EPOCHS = 3
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01

# LoRA configuration.
LORA_CONFIG_BART = {"r": 8, "alpha": 32, "dropout": 0.05}
LORA_CONFIG_LED = {"r": 4, "alpha": 16, "dropout": 0.05}

# Batch configuration.
BATCH_CONFIG_BART = {"batch_size": 4, "gradient_accumulation_steps": 4}
BATCH_CONFIG_LED = {"batch_size": 1, "gradient_accumulation_steps": 16}

# Profiling configuration.
PROFILE_SUBSET_BART = 500
PROFILE_SUBSET_LED = 80
PROFILE_SOURCE_LENGTH_LED = 4096

# TIS bins.
TIS_BINS = {
    "low": (0, 0.2),
    "medium": (0.2, 0.5),
    "high": (0.5, 1.0),
}

print("Configuration initialized.")
print(f"Dataset root: {DATASET_ROOT}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")
print(f"Pipeline version: {PIPELINE_VERSION}")

*This cell defines two small helper functions used later in the notebook: a simple JSON checkpoint manager for long-running steps and a utility to print consistent section banners — presented here exactly as-is.*

In [ ]:
# Utility helpers: checkpoint manager and section banner
# Provides a minimal JSON checkpoint loader/saver for long-running steps and a
# small helper to print consistent section banners used throughout the notebook.

def manage_checkpoint(path: Path, data: dict | None = None, mode: str = 'load'):
    """Save or load notebook state for long-running steps."""
    if mode == 'save' and data is not None:
        with open(path, 'w', encoding='utf-8') as handle:
            json.dump(data, handle, indent=2)
    elif mode == 'load' and path.exists():
        with open(path, 'r', encoding='utf-8') as handle:
            return json.load(handle)
    return None

def section_banner(title: str) -> None:
    """Print a short, consistent section banner."""
    print()
    print(title)
    print('=' * max(len(title), 60))

## **Phase 2: Dataset Loading & Profiling**

*Loads the PubMed train/validation/test splits, constructs deterministic sampled subsets for interactive profiling, and computes split-level token statistics (counts, averages, medians) to quantify truncation risk. Produces summary tables and graphical overviews for reporting.*

In [ ]:
# Dataset loading and deterministic sampling
# Loads train/validation/test splits from the dataset root and applies a
# deterministic sampling strategy (seeded) so notebook runs are reproducible.

section_banner("Dataset loading")

train_records_full = load_pubmed_split(DATASET_ROOT, split="train")
val_records_full = load_pubmed_split(DATASET_ROOT, split="validation")
test_records_full = load_pubmed_split(DATASET_ROOT, split="test")

# Use deterministic sampling for reproducibility in the notebook.
train_records = sample_records(train_records_full, TRAIN_SAMPLES, seed=42)
val_records = sample_records(val_records_full, VAL_SAMPLES, seed=42)
test_records = sample_records(test_records_full, TEST_SAMPLES, seed=42)

print(f"Train: {len(train_records)} records")
print(f"Validation: {len(val_records)} records")
print(f"Test: {len(test_records)} records")

In [ ]:
# Compute dataset statistics and display
import pandas as pd
import numpy as np
from IPython.display import display, HTML

stats_train = dataset_statistics(train_records)
stats_val = dataset_statistics(val_records)
stats_test = dataset_statistics(test_records)

article_medians = {
    "train": np.median([token_count(record.article) for record in train_records]) if train_records else 0,
    "validation": np.median([token_count(record.article) for record in val_records]) if val_records else 0,
    "test": np.median([token_count(record.article) for record in test_records]) if test_records else 0,
}

split_summary_df = pd.DataFrame(
    [
        {
            "split": "Train",
            "records": stats_train["records"],
            "avg_article_tokens": stats_train["avg_article_tokens"],
            "avg_abstract_tokens": stats_train["avg_abstract_tokens"],
            "median_article_tokens": article_medians['train'],
        },
        {
            "split": "Validation",
            "records": stats_val["records"],
            "avg_article_tokens": stats_val["avg_article_tokens"],
            "avg_abstract_tokens": stats_val["avg_abstract_tokens"],
            "median_article_tokens": article_medians['validation'],
        },
        {
            "split": "Test",
            "records": stats_test["records"],
            "avg_article_tokens": stats_test["avg_article_tokens"],
            "avg_abstract_tokens": stats_test["avg_abstract_tokens"],
            "median_article_tokens": article_medians['test'],
        },
    ]
)

# Save dataset metadata table for artifacts.
dataset_metadata_path_csv = REPORTS_DIR / "dataset_split_metadata.csv"
dataset_metadata_path_json = REPORTS_DIR / "dataset_split_metadata.json"
split_summary_df.to_csv(dataset_metadata_path_csv, index=False)
split_summary_df.to_json(dataset_metadata_path_json, orient='records', indent=2)

print(f"Saved dataset metadata to: {dataset_metadata_path_csv} and {dataset_metadata_path_json}")

# Display HTML table for reports and interactive review
display(HTML(split_summary_df.to_html(index=False, classes='table table-striped')))

In [ ]:
# Clean, separate figure: split sizes only.
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.container import BarContainer

# Apply global presentation aesthetic
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'Liberation Serif', 'DejaVu Serif']
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'Liberation Serif', 'DejaVu Serif']

plt.figure(figsize=(10, 6))
colors = ["#3498db", "#e67e22", "#2ecc71"]
ax = sns.barplot(data=split_summary_df, x="split", y="records", palette=colors)
ax.set_title("Dataset Split Sizes", fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel("Split", fontsize=14)
ax.set_ylabel("Number of Records", fontsize=14)

# Add labels
for container in ax.containers:
    if isinstance(container, BarContainer):
        ax.bar_label(container, padding=3, fontsize=12, fmt="%d")

sns.despine()
plt.tight_layout()

split_fig_path = REPORTS_DIR / "dataset_split_records.png"
plt.savefig(split_fig_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Truncation impact diagnostics for BART-large
# Computes the expected truncation ratio and approximate content loss given
# the average article length vs. the model's context window; prints caveats.

truncation_ratio = max(0, (stats_train['avg_article_tokens'] - CUTOFF_TOKENS_BART) / stats_train['avg_article_tokens'])
content_loss_pct = truncation_ratio * 100

print("BART-large truncation impact")
print(f"Average article length: {stats_train['avg_article_tokens']:.0f} tokens")
print(f"Context window: {CUTOFF_TOKENS_BART} tokens")
print(f"Truncation ratio: {truncation_ratio:.2%}")
print(f"Average content loss: {content_loss_pct:.1f}%")
print("Caveat: right-side truncation tends to omit methodological and results sections, which are frequently summary-relevant.")

In [ ]:
# Clean, separate figure: average article vs abstract tokens by split.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.container import BarContainer

# Prepare melted dataframe for seaborn
df_melted = pd.melt(
    split_summary_df, 
    id_vars=['split'], 
    value_vars=['avg_article_tokens', 'avg_abstract_tokens'],
    var_name='Token Type', 
    value_name='Average Tokens'
)
df_melted['Token Type'] = df_melted['Token Type'].map({
    'avg_article_tokens': 'Avg Article Tokens',
    'avg_abstract_tokens': 'Avg Abstract Tokens'
})

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=df_melted, 
    x='split', 
    y='Average Tokens', 
    hue='Token Type',
    palette={'Avg Article Tokens': '#3498db', 'Avg Abstract Tokens': '#e74c3c'}
)

# Added pad=35 to push the title up and make room for the legend
ax.set_title('Average Article vs Abstract Tokens by Split', fontsize=16, fontweight='bold', pad=35)
ax.set_xlabel('Split', fontsize=14)
ax.set_ylabel('Average Tokens', fontsize=14)

for container in ax.containers:
    if isinstance(container, BarContainer):
        ax.bar_label(container, padding=3, fontsize=11, fmt='%.1f')

# Add BART limit line if variable exists
try:
    limit = CUTOFF_TOKENS_BART
    ax.axhline(y=limit, color='black', linestyle='--', linewidth=2, label=f'BART Limit ({limit})')
except NameError:
    pass

# Moved legend to the top center (Ablation-style layout)
plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=3, frameon=False, fontsize=12)
sns.despine()
plt.tight_layout()

plt.savefig(REPORTS_DIR / 'dataset_average_tokens.png', dpi=300, bbox_inches='tight')
plt.show()

## **Phase 3: Truncation Impact Score (TIS)**

*Definition of the TIS metric and the corpus-level statistics required to estimate information loss resulting from truncation to a fixed token budget.*

### **Phase 3a: Oracle extractive baseline (TF–IDF sentence selection)**

*An extractive TF–IDF sentence-selection oracle that selects highest-scoring sentences until a token budget is reached; used as an upper-bound truncation-aware baseline.*

In [ ]:
# Oracle extractive baseline (TF–IDF sentence selection)
# Implements a simple extractive oracle that ranks sentences by TF–IDF score
# and selects top sentences until a token budget is filled (or partially filled).
# Used as an upper-bound baseline for truncation-aware comparisons.

def oracle_extract_tfidf_sentences(
    article: str,
    max_tokens: int = 1024,
    corpus_statistics: CorpusStatistics | None = None,
) -> str:
    """Select the most informative sentences up to the token budget."""
    sentences = simple_sentence_split(article)
    doc_freq = corpus_statistics.document_frequencies if corpus_statistics else None
    corpus_size = corpus_statistics.corpus_size if corpus_statistics else None

    sentence_scores = []
    for sentence in sentences:
        total_length = token_count(sentence)
        score = sentence_tfidf_score(sentence, doc_freq, corpus_size) if total_length else 0.0
        sentence_scores.append((sentence, total_length, score))

    sorted_sentences = sorted(sentence_scores, key=lambda x: x[2], reverse=True)
    selected_sentences = []
    token_count_used = 0

    for sentence, length, _ in sorted_sentences:
        if token_count_used + length <= max_tokens:
            selected_sentences.append((sentences.index(sentence), sentence))
            token_count_used += length
        elif token_count_used < max_tokens:
            selected_sentences.append((sentences.index(sentence), sentence))
            break

    selected_sentences.sort(key=lambda x: x[0])
    return " ".join([sent for _, sent in selected_sentences])

print("BART-Oracle strategy implemented.")

In [ ]:
# LED model specification summary
# Prints the LED model's spec details (context window and attention properties)
# for quick reference when reading the notebook results.

print(f"{LED_SPEC.name.split('/')[-1].upper()} architecture")
print(f"Max context window: {LED_SPEC.max_source_length} tokens")
print(f"Attention complexity: {LED_SPEC.complexity}")
print(f"Attention mechanism: {LED_SPEC.mechanism_details}")
print(f"Initialization: {LED_SPEC.initialization}")

In [ ]:
# Build corpus statistics and compute sample TIS
# Prepares corpus-level statistics used by the TIS computation and runs a
# small sampled TIS calculation for a lightweight summary shown below.

section_banner("TIS framework")

from scripts.tis import compute_tis_batch

print("Building corpus statistics for TIS calculation...")
corpus_stats = build_corpus_statistics(record.article for record in train_records)
print(f"Corpus vocabulary size: {len(corpus_stats.document_frequencies)}")
print(f"Corpus size: {corpus_stats.corpus_size}")

sample_size = min(500, len(train_records))
sampled_records = random.sample(train_records, sample_size)
tis_results_sample = compute_tis_batch(sampled_records, cutoff_tokens=CUTOFF_TOKENS_BART, corpus_statistics=corpus_stats)
tis_summary = summarize_tis(tis_results_sample)

print("TIS summary")
import pandas as pd
from IPython.display import display, HTML
df = pd.DataFrame([{"Metric": k.capitalize(), "Value": v} for k, v in tis_summary.items()])
display(HTML(df.to_html(index=False, classes='table table-striped')))

### **Phase 3b: Full TIS Computation (test set)**

*Computes TIS for BART and LED on the full test split and caches results to avoid recomputation during iterative notebook development.*

In [ ]:
# Full TIS computation (test set) with checkpointing
# Computes TIS for BART and LED on the full test split and caches results to
# avoid recomputation during iterative notebook development.

print("Computing full TIS for the test set...")

tis_checkpoint_path = REPORTS_DIR / "tis_test_results.checkpoint.json"
cached_data = manage_checkpoint(tis_checkpoint_path, mode='load')

if cached_data:
    tis_results_test = [TISResult(**r) for r in cached_data['results']]
    print(f"Loaded {len(tis_results_test)} BART TIS results from checkpoint.")
else:
    tis_results_test = []
    for record in test_records:
        tis_result = compute_tis(record.article, record.abstract, CUTOFF_TOKENS_BART, corpus_stats)
        tis_results_test.append(tis_result)
    manage_checkpoint(tis_checkpoint_path, {'results': [r.__dict__ for r in tis_results_test]}, mode='save')
    print("BART TIS computation complete and checkpointed.")

print("Computing TIS for LED-large as a sanity check...")
led_checkpoint_path = REPORTS_DIR / "tis_led_results.checkpoint.json"
cached_led = manage_checkpoint(led_checkpoint_path, mode='load')

if cached_led:
    tis_results_test_led = [TISResult(**r) for r in cached_led['results']]
    print(f"Loaded {len(tis_results_test_led)} LED TIS results from checkpoint.")
else:
    tis_results_test_led = []
    for record in test_records:
        tis_result = compute_tis(record.article, record.abstract, CUTOFF_TOKENS_LED, corpus_stats)
        tis_results_test_led.append(tis_result)
    manage_checkpoint(led_checkpoint_path, {'results': [r.__dict__ for r in tis_results_test_led]}, mode='save')
    print("LED TIS computation complete and checkpointed.")

led_tis_values = [r.tis for r in tis_results_test_led]
print("LED TIS statistics")
print(f"Mean: {np.mean(led_tis_values):.6f}")
print(f"Max: {max(led_tis_values):.6f}")
print(f"Sanity check: {'PASSED' if max(led_tis_values) < 1e-4 else 'FAILED'}")

## **Phase 4: Model Fine-Tuning Configurations**

*LoRA configuration and training wrapper setup for BART and LED fine-tuning; configuration objects created here are recorded in the final report.*

In [ ]:
# LoRA configuration summary
import pandas as pd
from IPython.display import display, HTML

section_banner("LoRA configuration")

data = []
for param, (bart_val, led_val) in {
    "LoRA Rank (r)": (LORA_CONFIG_BART['r'], LORA_CONFIG_LED['r']),
    "LoRA Alpha (α)": (LORA_CONFIG_BART['alpha'], LORA_CONFIG_LED['alpha']),
    "LoRA Dropout": (LORA_CONFIG_BART['dropout'], LORA_CONFIG_LED['dropout']),
    "Target Sequence Length": (f"{TARGET_SUMMARY_LENGTH} tokens", f"{TARGET_SUMMARY_LENGTH} tokens"),
}.items():
    data.append({"Parameter": param, "BART-large Settings": str(bart_val), "LED-large Settings": str(led_val)})

df = pd.DataFrame(data)
display(HTML(df.to_html(index=False, classes='table table-striped')))

In [ ]:
# LoRA configuration object factory and construction
# Small helper that builds the LoRA config dicts consumed by the training wrapper.

def create_lora_config(rank: int, alpha: int, dropout: float):
    """Create a LoRA configuration for sequence-to-sequence fine-tuning."""
    return {
        "rank": rank,
        "alpha": alpha,
        "dropout": dropout,
        "target_modules": ["q_proj", "v_proj"],
        "bias": "none",
        "task_type": "SEQ_2_SEQ_LM",
    }

lora_config_bart = create_lora_config(
    LORA_CONFIG_BART['r'],
    LORA_CONFIG_BART['alpha'],
    LORA_CONFIG_BART['dropout'],
)

lora_config_led = create_lora_config(
    LORA_CONFIG_LED['r'],
    LORA_CONFIG_LED['alpha'],
    LORA_CONFIG_LED['dropout'],
)

print("LoRA configuration objects created.")

### **Phase 4a: Models & limits**

*Declare the primary model specifications used for experiments and the extractive oracle baseline used for truncation-aware comparisons.*

In [ ]:
# Model architecture summary table
import pandas as pd
from IPython.display import display, HTML

section_banner("Model architectures")

oracle_spec = {
    "name": "BART-Oracle",
    "max_tokens": BART_SPEC.max_source_length,
    "params": BART_SPEC.parameters,
    "attention": BART_SPEC.attention_type,
    "role": "Upper-bound truncation baseline",
}

data = []
for spec in [BART_SPEC, LED_SPEC]:
    data.append({
        "Model": spec.name.split('/')[-1],
        "Max tokens": spec.max_source_length,
        "Parameters": spec.parameters,
        "Attention": spec.attention_type,
        "Role": spec.role
    })
data.append({
    "Model": oracle_spec['name'],
    "Max tokens": oracle_spec['max_tokens'],
    "Parameters": oracle_spec['params'],
    "Attention": oracle_spec['attention'],
    "Role": oracle_spec['role']
})
df = pd.DataFrame(data)
display(HTML(df.to_html(index=False, classes='table table-striped')))

### **Phase 4b: Training setup**

### Profiling subsets and length capping

Construct deterministic subsets for runtime and memory profiling. For LED, long articles are truncated to a fixed token length (PROFILE_SOURCE_LENGTH_LED) to emulate long-context behavior while maintaining tractable execution for interactive sessions. These subsets are intended for approximate throughput and GPU memory measurements and are not appropriate for final model training.

In [ ]:
# Profiling subsets and source-length capping
# Prepare deterministic profiling subsets for BART and LED and create a
# capped LED subset that limits source tokens for runtime/memory profiling.

section_banner("Profiling and runtime estimate")

profile_bart_subset = random.sample(train_records, min(PROFILE_SUBSET_BART, len(train_records)))
profile_led_subset = random.sample(train_records, min(PROFILE_SUBSET_LED, len(train_records)))

profile_led_subset_capped = [
    DatasetRecord(
        article=record.article if token_count(record.article) <= PROFILE_SOURCE_LENGTH_LED
                else " ".join(simple_tokenize(record.article)[:PROFILE_SOURCE_LENGTH_LED]),
        abstract=record.abstract,
        row_index=record.row_index,
    )
    for record in profile_led_subset
]

print(f"BART subset: {len(profile_bart_subset)} examples")
print(f"LED subset: {len(profile_led_subset_capped)} examples")
print(f"LED source length cap: {PROFILE_SOURCE_LENGTH_LED} tokens")
print(f"Training target examples: {TRAIN_SAMPLES}")

In [ ]:
# Training metadata and configuration object
# Construct a small metadata dict summarizing key training hyperparameters and
# LoRA/batch configs — useful for experiment traceability and passing to runners.

training_metadata = {
    "timestamp": datetime.now().isoformat(),
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "batch_configs": {"bart": BATCH_CONFIG_BART, "led": BATCH_CONFIG_LED},
    "lora_configs": {"bart": lora_config_bart, "led": lora_config_led},
}

print("Training metadata prepared.")

In [ ]:
# BART fine-tuning (calls training wrapper)
# Runs the parameter-efficient fine-tuning pipeline for BART using the
# consolidated `run_training_pipeline` helper. Wrapped in try/except to
# allow notebook iteration without stopping on training errors.

section_banner("BART fine-tuning")

output_dir_bart = FINE_TUNED_MODELS_DIR / "bart_large"

try:
    run_training_pipeline(
        model_name=BART_SPEC.name,
        train_records=train_records,
        val_records=val_records,
        output_dir=output_dir_bart,
        epochs=EPOCHS,
        is_led=False,
        resume_from_checkpoint=True,
    )
except Exception as exc:
    print(f"BART training failed: {exc}")

In [ ]:
# LED fine-tuning (calls training wrapper)
# Runs the parameter-efficient fine-tuning pipeline for LED (long-context)
# with a larger source length; exceptions are caught to keep notebook flow.

section_banner("LED fine-tuning")

output_dir_led = FINE_TUNED_MODELS_DIR / "led_large"

try:
    run_training_pipeline(
        model_name=LED_SPEC.name,
        train_records=train_records,
        val_records=val_records,
        output_dir=output_dir_led,
        epochs=EPOCHS,
        is_led=True,
        max_source_length=4096,
        resume_from_checkpoint=False,
    )
except Exception as exc:
    print(f"LED training failed: {exc}")

In [ ]:
# Sensitivity analysis
import json
import pandas as pd
from IPython.display import display, HTML

print("TIS threshold sensitivity analysis")
print("=" * 90)

analysis_path = REPORTS_DIR / "tis_analysis.json"
with open(analysis_path, "r", encoding="utf-8") as handle:
    data = json.load(handle)
    tis_values_full = data["bart_tis_values"]
    rouge2_gaps = data["rouge2_gaps"]

thresholds_to_test = [
    (0.1, 0.4), (0.2, 0.5), (0.2, 0.6),
    (0.3, 0.6), (0.3, 0.7), (0.4, 0.7),
]

results = []
for low_up, med_up in thresholds_to_test:
    low_gaps, med_gaps, high_gaps = [], [], []

    for tis, gap in zip(tis_values_full, rouge2_gaps):
        if tis < low_up: low_gaps.append(gap)
        elif tis <= med_up: med_gaps.append(gap)
        else: high_gaps.append(gap)

    mean_low = sum(low_gaps) / len(low_gaps) if low_gaps else 0
    mean_med = sum(med_gaps) / len(med_gaps) if med_gaps else 0
    mean_high = sum(high_gaps) / len(high_gaps) if high_gaps else 0

    is_monotonic = mean_low < mean_med < mean_high
    results.append({
        "Thresholds (Low, Med)": f"Low < {low_up:.1f}, High > {med_up:.1f}",
        "Low Gap": mean_low,
        "Med Gap": mean_med,
        "High Gap": mean_high,
        "Monotonic?": "YES" if is_monotonic else "NO"
    })

df = pd.DataFrame(results)
display(HTML(df.to_html(index=False, classes='table table-striped')))

## **Phase 5: TIS-stratified evaluation**

*Partition the test set into TIS difficulty bins and compute per-bin evaluation metrics to analyze performance as a function of truncation difficulty.*

### **Phase 5a: Partitioning test samples by TIS**

*Group test set documents into low, medium, and high difficulty bins using the Truncation Impact Score (TIS) for stratified evaluation and per-bin artifact generation.*

In [ ]:
test_records_with_tis = list(zip(test_records, tis_results_test))

tis_bins = {
    "low": [],
    "medium": [],
    "high": [],
}

for record, tis_result in test_records_with_tis:
    if tis_result.tis < 0.2:
        tis_bins["low"].append((record, tis_result))
    elif tis_result.tis <= 0.5:
        tis_bins["medium"].append((record, tis_result))
    else:
        tis_bins["high"].append((record, tis_result))

import pandas as pd
from IPython.display import display, HTML

print("TIS bin distribution")
dist_data = []
for bin_name, bin_data in tis_bins.items():
    share = (len(bin_data) / len(test_records_with_tis) * 100) if test_records_with_tis else 0
    dist_data.append({"Bin": bin_name.upper(), "Records": len(bin_data), "Share (%)": f"{share:.1f}%"})
display(HTML(pd.DataFrame(dist_data).to_html(index=False, classes='table table-striped')))

print("Representative TIS values")
rep_data = []
for bin_name, bin_data in tis_bins.items():
    if bin_data:
        import numpy as np
        tis_vals = [tis.tis for _, tis in bin_data]
        rep_data.append({
            "Bin": bin_name.upper(), 
            "Min": min(tis_vals), 
            "Mean": np.mean(tis_vals), 
            "Max": max(tis_vals)
        })
display(HTML(pd.DataFrame(rep_data).to_html(index=False, classes='table table-striped')))

In [ ]:
# Save summary tables to artifacts
pd.DataFrame(dist_data).to_json(REPORTS_DIR / "tis_bin_distribution.json", orient="records", indent=2)
pd.DataFrame(rep_data).to_json(REPORTS_DIR / "tis_representative_values.json", orient="records", indent=2)
print("Saved bin distributions and representative values to JSON artifacts.")

### Batched inference and generation

Load the fine-tuned LoRA adapters for BART and LED and generate summaries for the evaluation subset using batched generation. The summarizers accept a list of source documents and return generated summaries. Progress bars (tqdm) provide observable progress during longer runs; batch sizes are chosen according to expected GPU memory constraints.

In [ ]:
# Batched inference and generation (BART & LED) with Isolated Checkpointing
import tqdm
from scripts.text import oracle_summary

# Align to 1,500 samples
test_records_eval = sample_records(test_records_full, 1500, seed=42)

# Define individual cache paths
bart_cache_path = REPORTS_DIR / "bart_preds.checkpoint.json"
led_cache_path = REPORTS_DIR / "led_preds.checkpoint.json"
oracle_cache_path = REPORTS_DIR / "oracle_preds.checkpoint.json"

# ==========================================
# 1. BART INFERENCE
# ==========================================
cached_bart = manage_checkpoint(bart_cache_path, mode='load')
if cached_bart and len(cached_bart.get("preds", [])) == len(test_records_eval):
    print(f"Loaded {len(cached_bart['preds'])} BART predictions from cache.")
    bart_predictions = cached_bart["preds"]
else:
    print("Generating BART predictions...")
    bart_summarizer = HuggingFaceSummarizer(str(FINE_TUNED_MODELS_DIR / "bart_large"), max_source_length=1024)
    bart_predictions = bart_summarizer.generate_batch(
        [record.article for record in test_records_eval],
        batch_size=8,
        progress_bar=tqdm.tqdm(total=len(test_records_eval))
    )
    manage_checkpoint(bart_cache_path, {"preds": bart_predictions}, mode='save')


# ==========================================
# 2. LED INFERENCE
# ==========================================
cached_led = manage_checkpoint(led_cache_path, mode='load')
if cached_led and len(cached_led.get("preds", [])) == len(test_records_eval):
    print(f"Loaded {len(cached_led['preds'])} LED predictions from cache.")
    led_predictions = cached_led["preds"]
else:
    print("Generating LED predictions...")
    led_summarizer = HuggingFaceSummarizer(str(FINE_TUNED_MODELS_DIR / "led_large"), max_source_length=4096)
    led_predictions = led_summarizer.generate_batch(
        [record.article for record in test_records_eval],
        batch_size=4,
        progress_bar=tqdm.tqdm(total=len(test_records_eval))
    )
    manage_checkpoint(led_cache_path, {"preds": led_predictions}, mode='save')


# ==========================================
# 3. ORACLE INFERENCE
# ==========================================
cached_oracle = manage_checkpoint(oracle_cache_path, mode='load')
if cached_oracle and len(cached_oracle.get("preds", [])) == len(test_records_eval):
    print(f"Loaded {len(cached_oracle['preds'])} Oracle predictions from cache.")
    oracle_predictions = cached_oracle["preds"]
else:
    print("Generating Oracle Baselines...")
    oracle_predictions = [oracle_summary(r.article, r.abstract) for r in test_records_eval]
    manage_checkpoint(oracle_cache_path, {"preds": oracle_predictions}, mode='save')


print("All inference complete and cached!")


### Alignment check and TIS recomputation

Verify that the number of generated predictions, test records, and TIS entries are consistent. If counts are mismatched for the target evaluation size, recompute TIS for the exact sample used in evaluation to guarantee correct alignment for metric computations.

In [ ]:
# Alignment check and TIS recomputation if needed
# Ensures the predictions, records, and TIS arrays are aligned for final
# evaluation. If counts mismatch, recompute TIS for the selected evaluation
# sample to guarantee correct metric aggregation.

target_count = 1500
test_records = sample_records(test_records_full, target_count, seed=42)

print("Aligning lengths for final evaluation")
print(f"Records: {len(test_records)}")

if 'tis_results_test' not in locals() or len(tis_results_test) != target_count:
    print("Recomputing TIS for 1,500 samples...")
    corpus_stats = build_corpus_statistics([r.article for r in test_records])
    tis_results_test = compute_tis_batch(test_records, cutoff_tokens=1024, corpus_statistics=corpus_stats)
else:
    print("TIS results are already aligned.")

print(f"BART predictions: {len(bart_predictions)}")
print(f"LED predictions: {len(led_predictions)}")

if len(bart_predictions) != target_count or len(led_predictions) != target_count:
    print("Predictions are not aligned yet. Re-run the inference cell.")
else:
    print("All variables are aligned for evaluation.")

In [ ]:
# TIS-stratified evaluation wrapper
# Call the evaluation helper to compute per-bin and overall metrics grouped by
# pre-defined TIS thresholds. The helper returns a structured report used for
# downstream visualization and artifact writing.

section_banner("TIS-stratified evaluation")

report = evaluate_by_tis_bins(
    records=test_records,
    tis_results=tis_results_test,
    bart_predictions=bart_predictions,
    led_predictions=led_predictions,
    thresholds=BinThresholds(low_upper=0.2, medium_upper=0.5),
)

print("Evaluation report created.")

In [ ]:
# Sanity checks on evaluation artifacts
# Simple checks to ensure TIS values and report metrics look sensible before
# persisting artifacts; prints a few aggregate values for quick inspection.

section_banner("Sanity checks")

max_tis_val = max([r.tis for r in tis_results_test]) if tis_results_test else 0
print(f"Max TIS at 1k cutoff: {max_tis_val:.4f}")

overall_bart = report['overall']['bart']['rouge_2']
overall_led = report['overall']['led']['rouge_2']
print(f"Overall BART ROUGE-2: {overall_bart:.4f}")
print(f"Overall LED ROUGE-2: {overall_led:.4f}")

print("Sanity checks complete.")

In [ ]:
# Sample model outputs for qualitative inspection

if 'test_records_eval' not in globals():
    if 'test_records' in globals():
        test_records_eval = test_records
    elif 'test_records_full' in globals():
        test_records_eval = sample_records(test_records_full, 1500, seed=42)
    else:
        test_records_eval = []

if 'bart_predictions' not in globals():
    cached_bart = manage_checkpoint(REPORTS_DIR / 'bart_preds.checkpoint.json', mode='load')
    bart_predictions = cached_bart.get('preds', []) if cached_bart else []

if 'led_predictions' not in globals():
    cached_led = manage_checkpoint(REPORTS_DIR / 'led_preds.checkpoint.json', mode='load')
    led_predictions = cached_led.get('preds', []) if cached_led else []

if 'oracle_predictions' not in globals():
    cached_oracle = manage_checkpoint(REPORTS_DIR / 'oracle_preds.checkpoint.json', mode='load')
    oracle_predictions = cached_oracle.get('preds', []) if cached_oracle else []

sample_count = min(2, len(test_records_eval), len(bart_predictions), len(led_predictions), len(oracle_predictions))

if sample_count == 0:
    print('Sample outputs are not available yet; run the batched inference cell first or load the cached prediction artifacts.')
else:
    for i in range(sample_count):
        print(f"Sample {i + 1}")
        print(f"Reference: {test_records_eval[i].abstract[:200]}...")
        print(f"BART: {bart_predictions[i][:300]}...")
        print(f"LED: {led_predictions[i][:300]}...")
        print(f"Oracle: {oracle_predictions[i][:300]}...")
        print("-" * 60)

## **Phase 6: Artifacts**

*Write evaluation reports, per-bin summaries, TIS analyses, and visualization files to the artifacts directory for reproducible reporting.*

In [ ]:
# Models metadata configuration for artifact reports
# Collects model identifiers and relevant specs into a single dict that will
# be persisted as part of the experiment report for reproducibility.

models_config = {
    "bart": {
        "name": BART_SPEC.name,
        "max_context": BART_SPEC.max_source_length,
        "complexity": BART_SPEC.complexity,
        "role": BART_SPEC.role,
    },
    "led": {
        "name": LED_SPEC.name,
        "max_context": LED_SPEC.max_source_length,
        "complexity": LED_SPEC.complexity,
        "role": LED_SPEC.role,
    },
}

print("Model configuration collected.")

In [ ]:
# Persist evaluation reports and artifacts
# Writes the evaluation report, per-bin summaries, and TIS analyses to disk
# in JSON/CSV formats to the reports directory for reproducibility.

report_path = REPORTS_DIR / "full_pipeline_report.json"
with open(report_path, 'w', encoding='utf-8') as fh:
    json.dump(report, fh, indent=2)

print(f"Saved evaluation report to: {report_path}")

In [ ]:
# Save per-bin summaries and TIS analysis artifacts
# Save stratified metrics and TIS analysis outputs (CSV/JSON) for later plotting
# and for inclusion in the final report.

bin_csv = REPORTS_DIR / "tis_bin_metrics.csv"
bin_json = REPORTS_DIR / "tis_bin_metrics.json"

# Convert report structure into a DataFrame-friendly table (implementation depends on report format).
# Here we assume `report['bins']` is a mapping of bin names to metric dicts.

bins_data = []
for bin_name, metrics in report.get('bins', {}).items():
    row = {'bin': bin_name}
    row.update(metrics.get('rouge_2', {}))
    bins_data.append(row)

if bins_data:
    pd.DataFrame(bins_data).to_csv(bin_csv, index=False)
    pd.DataFrame(bins_data).to_json(bin_json, orient='records', indent=2)
    print(f"Saved bin summaries to {bin_csv} and {bin_json}")
else:
    print("No bin summaries available to save.")

## **Phase 7: Notebook summary**

*Consolidated list of produced artifacts and suggested follow-up experiments for reproducibility and extended analysis.*

In [ ]:
print("=" * 80)
print("Execution summary:")
print(f"Artifacts directory: {ARTIFACTS_DIR}")
print(f"Reports directory: {REPORTS_DIR}")
print(f"Full pipeline report: {REPORTS_DIR / 'full_pipeline_report.json'}")
print("=" * 80)

In [ ]:
# Recompute oracle metrics (extractive baseline)
# Compute aggregate metrics for the extractive oracle baseline used in
# ablation and comparison tables; results are persisted for later plotting.

# Recompute oracle metrics before the report and plots use them.
print("Calculating oracle baseline metrics...")
oracle_metrics = aggregate_metric_sets([
    summary_metrics(prediction, record.abstract)
    for prediction, record in zip(oracle_predictions, test_records_eval)
])

print("Oracle metric computation complete.")

In [ ]:
# Persist oracle baseline metrics
# Save the computed oracle baseline metrics to the reports directory for
# reproducibility and later inclusion in ablation figures.

oracle_report_path = REPORTS_DIR / "oracle_metrics.json"

with open(oracle_report_path, "w", encoding="utf-8") as handle:
    json.dump(oracle_metrics, handle, indent=2)

print(f"Saved oracle metrics to: {oracle_report_path}")

In [ ]:
# ======================================================
# Training Loss History Diagram
# ======================================================
bart_train = globals().get('bart_train', [])
led_train = globals().get('led_train', [])
bart_eval = globals().get('bart_eval', [])
led_eval = globals().get('led_eval', [])

if bart_train and led_train and bart_eval and led_eval:
    plt.figure(figsize=(10, 6))
    plt.plot(bart_train, label='BART Train', color='#3498db', alpha=0.3)
    plt.plot(led_train, label='LED Train', color='#e74c3c', alpha=0.3)
    
    # Plot evaluation points spaced across the training steps
    plt.plot(np.linspace(0, len(bart_train), len(bart_eval)), bart_eval, 'o-', color='#2980b9', label='BART Eval', linewidth=2)
    plt.plot(np.linspace(0, len(led_train), len(led_eval)), led_eval, 's-', color='#c0392b', label='LED Eval', linewidth=2)
    
    plt.title('Training and Validation Loss History', fontsize=16, fontweight='bold', pad=15)
    plt.xlabel('Training Steps', fontsize=14)
    plt.ylabel('Loss', fontsize=14)
    
    plt.legend(frameon=True, shadow=True)
    sns.despine()
    plt.tight_layout()
    
    loss_fig_path = REPORTS_DIR / '1_loss_history.png'
    plt.savefig(loss_fig_path, dpi=300)
    plt.show()
else:
    print('Training loss history is not available in the current notebook state.')

In [ ]:
# Stratified ROUGE-2 performance by TIS bin.
bins = ['low', 'medium', 'high']
bart_r2 = [report['bins'][b]['bart']['rouge_2'] for b in bins]
led_r2 = [report['bins'][b]['led']['rouge_2'] for b in bins]

df_bin = pd.DataFrame({
    'Bin': ['Low', 'Medium', 'High'] * 2,
    'ROUGE-2': bart_r2 + led_r2,
    'Model': ['BART (1k)'] * 3 + ['LED (4k)'] * 3,
})

# Added explicit dimensions to widen the chart
plt.figure(figsize=(10, 6))

ax = sns.barplot(data=df_bin, x='Bin', y='ROUGE-2', hue='Model', palette={'BART (1k)': '#3498db', 'LED (4k)': '#e74c3c'})
add_bar_labels(ax, fmt='%.4f')
ax.set_ylim(0, max(max(bart_r2), max(led_r2)) * 1.15)

# Increased title pad to 35 to make room for the new legend
plt.title('Stratified ROUGE-2 Performance by Difficulty', fontsize=16, fontweight='bold', pad=35)
plt.xlabel('TIS Difficulty Bin', fontsize=14)
plt.ylabel('ROUGE-2 Score', fontsize=14)

# Updated the legend (horizontal, top-center, no frame)
plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=2, frameon=False, fontsize=12)

sns.despine()
plt.tight_layout()
plt.savefig('./artifacts/reports/2_stratified_performance.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Test set TIS distribution.
plt.figure(figsize=(10, 6))
sns.histplot(tis_values_full, bins=25, kde=True, color='teal', edgecolor='black', alpha=0.6)
plt.axvline(0.2, color='#27ae60', ls='--', linewidth=2, label='Low/Med threshold (0.2)')
plt.axvline(0.5, color='#c0392b', ls='--', linewidth=2, label='Med/High threshold (0.5)')
plt.title('PubMed Test Set TIS Distribution', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Truncation Impact Score (TIS)', fontsize=14)
plt.ylabel('Number of articles', fontsize=14)
plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.savefig('./artifacts/reports/3_tis_distribution.png', dpi=300)
plt.show()

In [ ]:
# TIS versus ROUGE-2 gap analysis.
correlation = report['overall']['pearson_tis_vs_rouge2_gap']
correlation = report['overall']['pearson_tis_vs_rouge2_gap']
scatter_df = pd.DataFrame({'TIS': tis_values_full, 'ROUGE2_Gap': rouge2_gaps})


def determine_relationship(gap):
    if gap > 0.001:
        return 'LED (4k) > BART (1k)'
    if gap < -0.001:
        return 'BART (1k) > LED (4k)'
    return 'Equivalent Performance'


scatter_df['Performance_Delta'] = scatter_df['ROUGE2_Gap'].apply(determine_relationship)

plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=scatter_df,
    x='TIS',
    y='ROUGE2_Gap',
    hue='Performance_Delta',
    palette={
        'LED (4k) > BART (1k)': '#27ae60',
        'BART (1k) > LED (4k)': '#c0392b',
        'Equivalent Performance': '#7f8c8d',
    },
    alpha=0.6,
    s=50,
    edgecolor='none',
)
sns.regplot(
    data=scatter_df,
    x='TIS',
    y='ROUGE2_Gap',
    scatter=False,
    color='black',
    line_kws={'linestyle': '--', 'linewidth': 2.5, 'label': f'Linear Trend (r={correlation:.4f})'},
)
plt.axhline(0, color='black', linewidth=1.5, alpha=0.8)
plt.title('TIS vs ROUGE-2 Gap Analysis', fontsize=16, fontweight='bold', pad=35)
plt.xlabel('Document Difficulty (TIS Score)', fontsize=14)
plt.ylabel('ROUGE-2 Gap (LED minus BART)', fontsize=14)
plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=4, frameon=False, fontsize=12)
sns.despine()
plt.tight_layout()
plt.savefig('./artifacts/reports/4_scatter_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Ablation study: BART vs LED vs Oracle.
metrics_labels = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
bart_scores = [report['overall']['bart']['rouge_1'], report['overall']['bart']['rouge_2'], report['overall']['bart']['rouge_l']]
led_scores = [report['overall']['led']['rouge_1'], report['overall']['led']['rouge_2'], report['overall']['led']['rouge_l']]
oracle_scores = [oracle_metrics['rouge_1'], oracle_metrics['rouge_2'], oracle_metrics['rouge_l']]

df_triple = pd.DataFrame({
    'Metric': metrics_labels * 3,
    'Score': bart_scores + led_scores + oracle_scores,
    'Model': ['BART (1k Baseline)'] * 3 + ['LED (4k Long-Context)'] * 3 + ['Oracle (1k Smart-Select)'] * 3,
})

plt.figure(figsize=(12, 7))
palette = {
    'BART (1k Baseline)': '#3498db',
    'LED (4k Long-Context)': '#e74c3c',
    'Oracle (1k Smart-Select)': '#f1c40f',
}
ax = sns.barplot(data=df_triple, x='Metric', y='Score', hue='Model', palette=palette)
for container in ax.containers:
    if isinstance(container, BarContainer):
        ax.bar_label(container, fmt='%.4f', padding=5, fontsize=11)
ax.set_ylim(0, max(max(bart_scores), max(oracle_scores)) * 1.25)
plt.title('Ablation Study: Architecture vs. Intelligent Truncation', fontsize=16, fontweight='bold', pad=25)
plt.xlabel('Evaluation Metric', fontsize=14)
plt.ylabel('Score', fontsize=14)
plt.legend(title='', loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=3, frameon=False, fontsize=12)
sns.despine()
plt.tight_layout()
plt.savefig('./artifacts/reports/5_oracle_ablation_comparison.png', dpi=300, bbox_inches='tight')
plt.show()